# Cruzamento Domínio x DFE/SAT SC

Este notebook pede os dois arquivos separadamente, compara as NF-es da DFE com as Entradas do Domínio e gera uma planilha com as notas faltantes e diferenças de valor.

**Envie primeiro o arquivo de Entradas do Domínio** e depois a planilha **DFE/SAT**.

In [ ]:
!pip install -q pandas openpyxl xlrd xlsxwriter

import re
import shutil
import subprocess
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import files

pd.set_option('display.max_columns', 120)

In [ ]:
def sem_acento(texto):
    texto = '' if pd.isna(texto) else str(texto)
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(ch for ch in texto if not unicodedata.combining(ch))
    texto = re.sub(r'\s+', ' ', texto).strip().lower()
    return texto


def nome_coluna(col):
    return re.sub(r'[^a-z0-9]+', '_', sem_acento(col)).strip('_')


def tornar_colunas_unicas(cols):
    resultado = []
    vistos = {}
    for col in cols:
        base = nome_coluna(col) or 'coluna'
        vistos[base] = vistos.get(base, 0) + 1
        resultado.append(base if vistos[base] == 1 else f'{base}_{vistos[base]}')
    return resultado


def normalizar_numero_nf(valor):
    if pd.isna(valor):
        return ''
    texto = str(valor).strip()
    if texto.endswith('.0'):
        texto = texto[:-2]
    digitos = re.sub(r'\D+', '', texto)
    if not digitos:
        return ''
    return str(int(digitos))


def normalizar_serie(valor):
    if pd.isna(valor):
        return ''
    texto = str(valor).strip()
    if texto.endswith('.0'):
        texto = texto[:-2]
    digitos = re.sub(r'\D+', '', texto)
    if not digitos:
        return texto.upper()
    return str(int(digitos))


def normalizar_valor(valor):
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, str):
        valor = valor.strip().replace('.', '').replace(',', '.') if ',' in valor else valor.strip()
    return pd.to_numeric(valor, errors='coerce')


def normalizar_data_excel(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_datetime(serie, errors='coerce', unit='D', origin='1899-12-30')
    return pd.to_datetime(serie, errors='coerce', dayfirst=True)


def achar_coluna(df, candidatos, obrigatoria=True):
    colunas_norm = {col: sem_acento(col).replace('_', ' ') for col in df.columns}
    for candidato in candidatos:
        cand = sem_acento(candidato).replace('_', ' ')
        for col, norm in colunas_norm.items():
            if cand == norm or cand in norm:
                return col
    if obrigatoria:
        raise ValueError(f'Nao encontrei coluna esperada. Procurei por: {candidatos}. Colunas disponiveis: {list(df.columns)}')
    return None


def escolher_primeiro_upload(mensagem):
    print(mensagem)
    enviados = files.upload()
    if not enviados:
        raise ValueError('Nenhum arquivo enviado.')
    return next(iter(enviados.keys()))


def instalar_libreoffice_se_precisar():
    executavel = shutil.which('libreoffice') or shutil.which('soffice')
    if executavel:
        return executavel
    print('Instalando LibreOffice para converter o .xls antigo do Dominio...')
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'libreoffice'], check=True)
    executavel = shutil.which('libreoffice') or shutil.which('soffice')
    if not executavel:
        raise RuntimeError('Nao consegui instalar/localizar o LibreOffice para converter o .xls.')
    return executavel


def converter_xls_para_xlsx(caminho):
    caminho = Path(caminho)
    saida_dir = Path('arquivos_convertidos')
    saida_dir.mkdir(exist_ok=True)
    destino = saida_dir / f'{caminho.stem}.xlsx'
    if destino.exists():
        return str(destino)

    executavel = instalar_libreoffice_se_precisar()
    comando = [executavel, '--headless', '--convert-to', 'xlsx', '--outdir', str(saida_dir), str(caminho)]
    processo = subprocess.run(comando, text=True, capture_output=True)
    if processo.returncode != 0 or not destino.exists():
        detalhe = (processo.stderr or processo.stdout or '').strip()
        raise RuntimeError(f'Falha ao converter {caminho.name} para .xlsx. {detalhe}')
    print(f'Arquivo convertido para leitura: {destino}')
    return str(destino)


def preparar_excel_para_leitura(caminho):
    try:
        pd.ExcelFile(caminho)
        return caminho
    except Exception as erro:
        if Path(caminho).suffix.lower() == '.xls':
            print(f'O arquivo {Path(caminho).name} nao foi lido diretamente pelo pandas/xlrd.')
            print('Vou converter para .xlsx e continuar a conferencia.')
            return converter_xls_para_xlsx(caminho)
        raise erro


def encontrar_aba_e_cabecalho(caminho, termos):
    caminho_leitura = preparar_excel_para_leitura(caminho)
    xls = pd.ExcelFile(caminho_leitura)
    melhor = None
    for aba in xls.sheet_names:
        bruto = pd.read_excel(caminho_leitura, sheet_name=aba, header=None, dtype=object)
        for idx in range(min(30, len(bruto))):
            linha = ' | '.join(sem_acento(v) for v in bruto.iloc[idx].tolist())
            pontos = sum(1 for termo in termos if sem_acento(termo) in linha)
            if melhor is None or pontos > melhor['pontos']:
                melhor = {'aba': aba, 'header_idx': idx, 'pontos': pontos, 'bruto': bruto}
    if melhor is None or melhor['pontos'] == 0:
        raise ValueError(f'Nao consegui detectar cabecalho em {caminho}.')
    return melhor


def carregar_entradas_dominio(caminho):
    info = encontrar_aba_e_cabecalho(caminho, ['nota', 'serie', 'fornecedor', 'valor'])
    bruto = info['bruto']
    cabecalho = bruto.iloc[info['header_idx']].tolist()
    df = bruto.iloc[info['header_idx'] + 1:].copy()
    df.columns = tornar_colunas_unicas(cabecalho)
    df = df.dropna(how='all').reset_index(drop=True)
    manter = []
    for col in df.columns:
        coluna_generica = bool(re.match(r'^coluna(_\d+)?$', str(col)))
        manter.append((not coluna_generica) or df[col].notna().any())
    df = df.loc[:, manter]

    col_nota = achar_coluna(df, ['nota'])
    col_serie = achar_coluna(df, ['serie'])
    col_valor = achar_coluna(df, ['valor contabil', 'valor_contabil', 'valor'])
    col_fornecedor = achar_coluna(df, ['fornecedor'], obrigatoria=False)
    col_data = achar_coluna(df, ['data'], obrigatoria=False)

    saida = pd.DataFrame()
    saida['dominio_numero'] = df[col_nota].map(normalizar_numero_nf)
    saida['dominio_serie'] = df[col_serie].map(normalizar_serie)
    saida['dominio_valor'] = df[col_valor].map(normalizar_valor).round(2)
    if col_fornecedor:
        saida['dominio_fornecedor'] = df[col_fornecedor]
    if col_data:
        saida['dominio_data'] = normalizar_data_excel(df[col_data])
    saida['dominio_aba_origem'] = info['aba']
    saida = saida[(saida['dominio_numero'] != '') & (saida['dominio_serie'] != '')].copy()
    return saida, df


def carregar_dfe(caminho):
    info = encontrar_aba_e_cabecalho(caminho, ['ChaveAcesso', 'NumeroDocumento', 'ValorTotalNota'])
    bruto = info['bruto']
    df = bruto.iloc[info['header_idx'] + 1:].copy()
    df.columns = [str(c).strip() for c in bruto.iloc[info['header_idx']].tolist()]
    df = df.dropna(how='all').reset_index(drop=True)

    col_chave = achar_coluna(df, ['ChaveAcesso', 'chave acesso'])
    col_numero = achar_coluna(df, ['NumeroDocumento', 'numero documento'])
    col_serie = achar_coluna(df, ['SerieDocumento', 'serie documento'])
    col_valor = achar_coluna(df, ['ValorTotalNota', 'valor total nota'])

    df['_dfe_chave'] = df[col_chave].astype(str).str.replace(r'\D+', '', regex=True)
    df['_dfe_numero'] = df[col_numero].map(normalizar_numero_nf)
    df['_dfe_serie'] = df[col_serie].map(normalizar_serie)
    df['_dfe_valor'] = df[col_valor].map(normalizar_valor).round(2)
    df['_linha_dfe_origem'] = df.index + info['header_idx'] + 2

    chave_valida = df['_dfe_chave'].str.fullmatch(r'\d{44}', na=False)
    df_ignoradas = df[~chave_valida].copy()
    if not df_ignoradas.empty:
        df_ignoradas.insert(0, 'MotivoIgnorado', 'ChaveAcesso ausente ou invalida')
        print(f'Linhas da DFE ignoradas por nao terem chave valida: {len(df_ignoradas)}')

    df = df[chave_valida].copy().reset_index(drop=True)
    return df, df_ignoradas


def cruzar(dfe, entradas, tolerancia=0.01):
    grupos = {}
    for _, row in entradas.iterrows():
        chave = (row['dominio_numero'], row['dominio_serie'])
        grupos.setdefault(chave, []).append(row.to_dict())

    status = []
    valor_dominio = []
    fornecedor_dominio = []
    diferenca = []

    for _, row in dfe.iterrows():
        chave = (row['_dfe_numero'], row['_dfe_serie'])
        candidatos = grupos.get(chave, [])
        if not candidatos:
            status.append('FALTANDO NAS ENTRADAS')
            valor_dominio.append(np.nan)
            fornecedor_dominio.append('')
            diferenca.append(np.nan)
            continue

        melhor = min(candidatos, key=lambda c: abs((c.get('dominio_valor') or 0) - (row['_dfe_valor'] or 0)))
        val_dom = melhor.get('dominio_valor')
        dif = round((row['_dfe_valor'] or 0) - (val_dom or 0), 2) if pd.notna(val_dom) and pd.notna(row['_dfe_valor']) else np.nan
        if pd.notna(dif) and abs(dif) <= tolerancia:
            status.append('OK')
        else:
            status.append('DIFERENCA DE VALOR')
        valor_dominio.append(val_dom)
        fornecedor_dominio.append(melhor.get('dominio_fornecedor', ''))
        diferenca.append(dif)

    resultado = dfe.copy()
    resultado.insert(0, 'StatusConferencia', status)
    resultado.insert(1, 'ValorDominio', valor_dominio)
    resultado.insert(2, 'DiferencaDfeMenosDominio', diferenca)
    resultado.insert(3, 'FornecedorDominio', fornecedor_dominio)
    return resultado


def salvar_relatorio(resultado, entradas, dfe_ignoradas=None, caminho_saida='relatorio_dominio_x_dfe.xlsx'):
    pendencias = resultado[resultado['StatusConferencia'] != 'OK'].copy()
    faltantes = resultado[resultado['StatusConferencia'] == 'FALTANDO NAS ENTRADAS'].copy()
    diferencas = resultado[resultado['StatusConferencia'] == 'DIFERENCA DE VALOR'].copy()
    qtd_ignoradas = 0 if dfe_ignoradas is None else len(dfe_ignoradas)

    guia_abas = pd.DataFrame([
        ['Leia_me', 'Cabecalho do arquivo. Explica a finalidade de cada aba e como usar o relatorio.'],
        ['Resumo', 'Mostra a quantidade total de documentos conferidos, notas OK, faltantes, diferencas e linhas ignoradas.'],
        ['Pendencias', 'Junta tudo que precisa de analise: notas faltantes nas Entradas e notas com diferenca de valor.'],
        ['Faltantes_para_baixar', 'Lista somente as NF-es da DFE que nao foram encontradas nas Entradas. Use a ChaveAcesso para baixar/importar as notas.'],
        ['Diferencas_valor', 'Lista notas encontradas no Dominio, mas com ValorTotalNota diferente do Valor Contabil das Entradas.'],
        ['DFE_com_status', 'Base completa da DFE com a coluna StatusConferencia para auditar o resultado linha a linha.'],
        ['Entradas_normalizadas', 'Base de Entradas do Dominio ja padronizada com numero, serie, valor, fornecedor e data.'],
        ['DFE_ignoradas_sem_chave', 'Aparece somente quando houver linhas sem ChaveAcesso valida de 44 digitos, geralmente totais ou rodapes.'],
    ], columns=['Aba', 'Para que serve'])

    resumo = pd.DataFrame([
        ['Total DFE', len(resultado)],
        ['OK', int((resultado['StatusConferencia'] == 'OK').sum())],
        ['Faltando nas Entradas', len(faltantes)],
        ['Diferenca de valor', len(diferencas)],
        ['Linhas DFE ignoradas sem chave valida', qtd_ignoradas],
        ['Total pendencias', len(pendencias)],
    ], columns=['Indicador', 'Quantidade'])

    colunas_prioritarias = [
        'StatusConferencia', 'MotivoIgnorado', 'ChaveAcesso', 'DataEmissao', 'NumeroDocumento', 'SerieDocumento',
        'ValorTotalNota', 'ValorDominio', 'DiferencaDfeMenosDominio', 'NomeEmitente',
        'CnpjOuCpfDoEmitente', 'FornecedorDominio', 'Situacao'
    ]
    def ordenar_cols(df):
        primeiras = [c for c in colunas_prioritarias if c in df.columns]
        resto = [c for c in df.columns if c not in primeiras and not c.startswith('_')]
        return df[primeiras + resto]

    with pd.ExcelWriter(caminho_saida, engine='xlsxwriter') as writer:
        guia_abas.to_excel(writer, sheet_name='Leia_me', index=False, header=False, startrow=4)
        resumo.to_excel(writer, sheet_name='Resumo', index=False)
        ordenar_cols(pendencias).to_excel(writer, sheet_name='Pendencias', index=False)
        ordenar_cols(faltantes).to_excel(writer, sheet_name='Faltantes_para_baixar', index=False)
        ordenar_cols(diferencas).to_excel(writer, sheet_name='Diferencas_valor', index=False)
        ordenar_cols(resultado).to_excel(writer, sheet_name='DFE_com_status', index=False)
        entradas.to_excel(writer, sheet_name='Entradas_normalizadas', index=False)
        if dfe_ignoradas is not None and not dfe_ignoradas.empty:
            ordenar_cols(dfe_ignoradas).to_excel(writer, sheet_name='DFE_ignoradas_sem_chave', index=False)

        workbook = writer.book
        titulo_fmt = workbook.add_format({'bold': True, 'font_size': 16, 'font_color': '#1F4E78'})
        texto_fmt = workbook.add_format({'text_wrap': True})
        cabecalho_fmt = workbook.add_format({'bold': True, 'bg_color': '#D9EAF7', 'border': 1})
        ws_leia = writer.sheets['Leia_me']
        ws_leia.write('A1', 'Relatorio Dominio x DFE/SAT', titulo_fmt)
        ws_leia.write('A2', 'Use este arquivo para identificar NF-es da DFE que nao estao lancadas nas Entradas do Dominio e diferencas de valor.', texto_fmt)
        ws_leia.write('A4', 'Aba', cabecalho_fmt)
        ws_leia.write('B4', 'Para que serve', cabecalho_fmt)
        ws_leia.set_column('A:A', 28)
        ws_leia.set_column('B:B', 120, texto_fmt)
        ws_leia.set_row(1, 36)

        for sheet_name, ws in writer.sheets.items():
            if sheet_name == 'Leia_me':
                ws.freeze_panes(4, 0)
                ws.autofilter(3, 0, 3 + len(guia_abas), 1)
                continue
            ws.freeze_panes(1, 0)
            ws.autofilter(0, 0, 0, 20)
            ws.set_column(0, 0, 24)
            ws.set_column(1, 12, 18)
        for sheet_name, ws in writer.sheets.items():
            ws.set_column(1, 1, 48)
            if sheet_name in ['Pendencias', 'Faltantes_para_baixar', 'DFE_com_status']:
                ws.set_column(1, 1, 48)

    return caminho_saida

In [ ]:
arquivo_entradas = escolher_primeiro_upload('1) Envie agora o arquivo ENTRADAS do Dominio (.xls ou .xlsx):')
arquivo_dfe = escolher_primeiro_upload('2) Envie agora a planilha DFE/SAT (.xlsx):')

entradas, entradas_original = carregar_entradas_dominio(arquivo_entradas)
dfe, dfe_ignoradas = carregar_dfe(arquivo_dfe)
resultado = cruzar(dfe, entradas)
saida = salvar_relatorio(resultado, entradas, dfe_ignoradas)

print('Conferencia concluida.')
print(resultado['StatusConferencia'].value_counts(dropna=False))
if not dfe_ignoradas.empty:
    print(f'Linhas ignoradas por falta de chave valida: {len(dfe_ignoradas)}')
files.download(saida)

## Conferência rápida

Depois de baixar o Excel gerado, abra a aba `Faltantes_para_baixar`. Ela contém as linhas da DFE que não estão nas Entradas e a coluna `ChaveAcesso` para baixar/importar as notas faltantes.